## Template Visual Odometry

In [7]:
import cv2
from PIL import Image
import glob
import numpy as np
import cv2 as cv2
from numpy.linalg import inv, pinv
import matplotlib.pyplot as plt
import time as t

In [13]:
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt


# -----------------------------------------------------------
# Camera intrinsics (given in the assignment)
# -----------------------------------------------------------
def getK():
    """
    Return the 3x3 camera intrinsic matrix K.
    """
    return np.array([[7.188560e+02, 0.000000e+00, 6.071928e+02],
                     [0.000000e+00, 7.188560e+02, 1.852157e+02],
                     [0.000000e+00, 0.000000e+00, 1.000000e+00]],
                    dtype=np.float64)


# -----------------------------------------------------------
# Utility: load and SORT stereo images
# -----------------------------------------------------------
def load_stereo_sequence(path_left="left/*.png", path_right="right/*.png"):
    """
    Load left/right images and ensure they are sorted by filename so
    we get the correct temporal order for visual odometry.
    """
    left_files = sorted(glob.glob(path_left))
    right_files = sorted(glob.glob(path_right))

    images_left = [cv2.imread(f, cv2.IMREAD_GRAYSCALE) for f in left_files]
    images_right = [cv2.imread(f, cv2.IMREAD_GRAYSCALE) for f in right_files]

    assert len(images_left) == len(images_right), "Left/right image count mismatch"
    return images_left, images_right


# -----------------------------------------------------------
# Feature detection + matching + stereo triangulation
# -----------------------------------------------------------

# SIFT detector (you can switch to ORB if SIFT not available)
sift = cv2.SIFT_create()
MATCH_NUM = 2000  # max number of best matches to keep


def extract_keypoints_and_triangulate(img_left, img_right, K, baseline):
    """
    1) Detect SIFT features in left and right images.
    2) Match them with FLANN + Lowe's ratio test.
    3) Take the best matches and triangulate 3D points.
    4) Return:
        - landmark_3D_cam : Nx3 points in *camera frame* of the left image
        - pts_left        : Nx2 2D pixel coordinates in the left image
    """

    # 1. SIFT detection
    kp1, des1 = sift.detectAndCompute(img_left, None)
    kp2, des2 = sift.detectAndCompute(img_right, None)

    # 2. FLANN-based matcher + Lowe ratio
    matcher = cv2.DescriptorMatcher_create(cv2.DescriptorMatcher_FLANNBASED)
    knn_matches = matcher.knnMatch(des1, des2, k=2)

    good_matches = []
    ratio_thresh = 0.7
    for m, n in knn_matches:
        if m.distance < ratio_thresh * n.distance:
            good_matches.append(m)

    # Keep only the best MATCH_NUM matches
    good_matches = sorted(good_matches, key=lambda x: x.distance)
    good_matches = good_matches[:MATCH_NUM]

    # Extract the corresponding 2D points
    pts_left = np.float32([kp1[m.queryIdx].pt for m in good_matches])   # Nx2
    pts_right = np.float32([kp2[m.trainIdx].pt for m in good_matches])  # Nx2

    # 3. Build projection matrices for rectified stereo pair
    #    Left camera:  [I | 0]
    #    Right camera: [I | (-baseline, 0, 0)^T]
    M_left = K @ np.hstack((np.eye(3), np.zeros((3, 1))))
    M_right = K @ np.hstack((np.eye(3), np.array([[-baseline, 0, 0]]).T))

    # 4. Triangulate in homogeneous coordinates
    #    cv2.triangulatePoints expects 2xN arrays
    pts_left_h = pts_left.T   # 2xN
    pts_right_h = pts_right.T # 2xN

    P_h = cv2.triangulatePoints(M_left, M_right, pts_left_h, pts_right_h)  # 4xN
    P_h /= P_h[3]  # normalize

    # Convert to Nx3 Euclidean coordinates in *left-camera frame*
    landmark_3D_cam = P_h[:3].T

    return landmark_3D_cam.astype(np.float32), pts_left.astype(np.float32)


# -----------------------------------------------------------
# KLT tracking between consecutive left images
# -----------------------------------------------------------
def featureTracking(prev_img, next_img, prev_points, world_points):
    """
    Track 2D points from prev_img to next_img using LK optical flow.

    prev_points : Nx2 2D pixel locations in prev_img
    world_points: Nx3 3D points in *world* coordinates that correspond
                  one-to-one to prev_points.

    Returns:
        world_points_tracked : Mx3  (subset where tracking succeeded)
        prev_points_tracked  : Mx2  (prev positions)
        next_points_tracked  : Mx2  (current positions)
    """

    lk_params = dict(winSize=(21, 21),
                     maxLevel=3,
                     criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
                               30, 0.01))

    # calcOpticalFlowPyrLK expects points as Nx1x2 float32
    prev_pts = prev_points.reshape(-1, 1, 2).astype(np.float32)

    next_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_img, next_img,
                                                   prev_pts, None, **lk_params)

    status = status.reshape(-1)
    good_mask = status == 1

    world_points_tracked = world_points[good_mask]
    prev_points_tracked = prev_points[good_mask]
    next_points_tracked = next_pts[good_mask].reshape(-1, 2)

    return world_points_tracked, prev_points_tracked, next_points_tracked


# -----------------------------------------------------------
# Main VO loop
# -----------------------------------------------------------
def run_visual_odometry(images_left, images_right, K):
    """
    Complete VO pipeline for the assignment.

    - Uses the first stereo pair to create initial 3D landmarks
      in the first camera frame (world frame).
    - Then iterates over subsequent images, each time:
        * tracks 2D features,
        * estimates pose with PnP (world 3D -> current 2D),
        * triangulates new stereo matches,
        * transforms new 3D points to world coords and adds them
          to the landmark set.

    Prints pose (position + axis-angle rotation) for each frame.
    """

    baseline = 0.54  # m

    # --- Initialization with the first stereo pair ---------------------
    left0 = images_left[0]
    right0 = images_right[0]

    # 3D points in cam0 frame (we define this frame as "world")
    landmark_3D_cam0, pts2D_left0 = extract_keypoints_and_triangulate(
        left0, right0, K, baseline
    )

    # World frame == cam0 frame at the start
    landmark_3D_world = landmark_3D_cam0.copy()  # Nx3
    ref_2D = pts2D_left0.copy()                  # Nx2, last-seen 2D positions

    # Pose of camera 0 in world coordinates:
    # R_wc = I, t_wc = 0 -> camera center at origin, no rotation
    R_wc = np.eye(3, dtype=np.float64)    # world -> cam0
    t_wc = np.zeros((3, 1), dtype=np.float64)

    # We will store pose as camera center in world coordinates
    # C_w = -R_wc^T t_wc
    cam_positions = []
    cam_orientations = []  # rvec (axis-angle) per frame

    # For frame 0:
    C_w0 = -R_wc.T @ t_wc
    cam_positions.append(C_w0.flatten())
    cam_orientations.append(np.zeros(3))  # no rotation

    prev_img = left0

    # --- Process subsequent frames ------------------------------------
    for i in range(1, len(images_left)):
        print(f"\n---------------- Frame {i} ----------------")
        cur_img = images_left[i]
        cur_img_R = images_right[i]

        # 1) Track existing 2D features from prev -> current
        if len(ref_2D) < 10:
            print("Too few tracked features, stopping.")
            break

        world_pts_tr, ref_2D_tr, cur_2D_tr = featureTracking(
            prev_img, cur_img, ref_2D, landmark_3D_world
        )

        if len(world_pts_tr) < 6:
            print("Too few correspondences for PnP, stopping.")
            break

        # 2) PnP: estimate absolute pose of current camera wrt world
        #    world_pts_tr : 3D in world coordinates
        #    cur_2D_tr    : 2D in current image
        object_points = world_pts_tr.reshape(-1, 1, 3).astype(np.float32)
        image_points = cur_2D_tr.reshape(-1, 1, 2).astype(np.float32)

        # We assume no distortion (last arg None)
        success, rvec, tvec, inliers = cv2.solvePnPRansac(
            object_points, image_points, K, None
        )

        if not success:
            print("PnP failed, stopping.")
            break

        # R_wc_i, t_wc_i: transform from world -> camera_i
        R_wc_i, _ = cv2.Rodrigues(rvec)
        t_wc_i = tvec

        # Camera center in world coords:
        # C_w = -R^T * t
        C_w_i = -R_wc_i.T @ t_wc_i  # 3x1

        # Store pose
        cam_positions.append(C_w_i.flatten())
        cam_orientations.append(rvec.flatten())

        # 3) Triangulate new stereo points in the *current camera frame*
        landmark_3D_cam_i, pts2D_left_i = extract_keypoints_and_triangulate(
            cur_img, cur_img_R, K, baseline
        )

        # 4) Transform new landmarks from camera frame -> world frame
        #    x_world = R_cw * x_cam + t_cw,
        #    where R_cw = R_wc^T,  t_cw = C_w_i
        R_cw_i = R_wc_i.T
        t_cw_i = C_w_i  # already 3x1

        # landmark_3D_cam_i is Nx3 -> convert to world
        pts_cam = landmark_3D_cam_i.T  # 3xN
        pts_world_new = (R_cw_i @ pts_cam) + t_cw_i  # 3xN
        pts_world_new = pts_world_new.T  # Nx3

        # 5) Update world landmarks and last-seen 2D positions
        landmark_3D_world = np.vstack([world_pts_tr, pts_world_new])
        ref_2D = np.vstack([cur_2D_tr, pts2D_left_i])

        # 6) Prepare for next iteration
        prev_img = cur_img

        # 7) Print current pose (position + axis-angle)
        print("Position (camera center in world) [x, y, z] =",
              C_w_i.flatten())
        print("Orientation (axis-angle rvec) [rx, ry, rz] =",
              rvec.flatten())

    return np.array(cam_positions), np.array(cam_orientations)


# -----------------------------------------------------------
# Run everything
# -----------------------------------------------------------
if __name__ == "__main__":
    K = getK()
    images_left, images_right = load_stereo_sequence("left/*.png", "right/*.png")

    cam_positions, cam_orientations = run_visual_odometry(
        images_left, images_right, K
    )

    print("\n===== Final pose =====")
    print("Frame index:", len(cam_positions) - 1)
    print("Position [x, y, z]:", cam_positions[-1])
    print("Axis-angle [rx, ry, rz]:", cam_orientations[-1])



---------------- Frame 1 ----------------
Position (camera center in world) [x, y, z] = [-0.00676033 -0.0161181   0.62023309]
Orientation (axis-angle rvec) [rx, ry, rz] = [0.00312597 0.00054966 0.0017696 ]

---------------- Frame 2 ----------------
Position (camera center in world) [x, y, z] = [-0.01074528 -0.02605375  1.21663253]
Orientation (axis-angle rvec) [rx, ry, rz] = [ 0.00593248  0.00309916 -0.00039322]

---------------- Frame 3 ----------------
Position (camera center in world) [x, y, z] = [-0.00565692 -0.03307847  1.78650453]
Orientation (axis-angle rvec) [rx, ry, rz] = [ 0.00551045  0.00822122 -0.0069884 ]

---------------- Frame 4 ----------------
Position (camera center in world) [x, y, z] = [-0.0125047  -0.048108    2.33540762]
Orientation (axis-angle rvec) [rx, ry, rz] = [ 0.00373713  0.01624528 -0.01094089]

---------------- Frame 5 ----------------
Position (camera center in world) [x, y, z] = [-0.03114988 -0.05966654  2.86568491]
Orientation (axis-angle rvec) [rx, r